In [119]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [120]:
result_path = Path("../thesis_results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", 
               # "soft-mrs-exponential", 
                "fw-mrs-temperature_old",  
                "fw-mrs-temperature-svm_old",
                "fw-mrs-temperature",  "fw-mrs-temperature-svm",
                "fw-mrs-temperature-svm_soft_threshold", "fw-mrs-temperature_soft_threshold", "mrs-forest_soft_threshold"
                ]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
# less_bias_strengths = ["0.1", "0.2", "0.3"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction"]
method_name_replacer = {"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                            # "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "mrs-forest": "MRS", "fw-mrs-temperature_old": "FW-MRS-RF-Abs", "fw-mrs-temperature-svm_old": "FW-MRS-SVM-Abs",
                        "fw-mrs-temperature-svm": "FW-MRS-SVM-Signed", "fw-mrs-temperature": "FW-MRS-RF-Signed",
                          "fw-mrs-temperature-svm_soft_threshold": "FW-MRS-SVM-DA", "fw-mrs-temperature_soft_threshold": "FW-MRS-RF-DA",
                          "mrs-forest_soft_threshold": "MRS-DA",
                          }

In [121]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [122]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.870831,0.010040,0.827021,0.016671,less_positive_class,0.1,0.00,0.000000
1,PSA,folktables_employment,0.867464,0.010494,0.823775,0.016330,less_positive_class,0.1,0.04,0.280000
2,KMM,folktables_employment,0.856957,0.012656,0.809232,0.019908,less_positive_class,0.1,0.00,0.000000
3,MRS,folktables_employment,0.870307,0.010000,0.826587,0.016147,less_positive_class,0.1,176.60,29.264996
4,FW-MRS-RF-Abs,folktables_employment,0.840812,0.012290,0.787853,0.020694,less_positive_class,0.1,149.80,29.865699
5,FW-MRS-SVM-Abs,folktables_employment,0.855381,0.014431,0.809600,0.021618,less_positive_class,0.1,242.80,32.575451
6,FW-MRS-RF-Signed,folktables_employment,0.873373,0.008801,0.831082,0.013719,less_positive_class,0.1,347.70,125.144756
7,FW-MRS-SVM-Signed,folktables_employment,0.868225,0.010563,0.824525,0.017159,less_positive_class,0.1,276.30,33.133216
8,Uniform,folktables_income,0.838550,0.012914,0.788787,0.018468,less_positive_class,0.1,0.00,0.000000
9,PSA,folktables_income,0.831214,0.013329,0.780784,0.019630,less_positive_class,0.1,0.12,0.430813


In [123]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset in datasets:
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.871\pm0.01$ & $0.839\pm0.01$ & $0.752\pm0.02$ & $0.988\pm0.01$ & $0.672\pm0.07$ & \\
	& PSA & $0.867\pm0.01$ & $0.831\pm0.01$ & $0.753\pm0.02$ & $0.988\pm0.01$ & $0.648\pm0.09$ & \\
	& KMM & $0.857\pm0.01$ & $0.82\pm0.01$ & $0.746\pm0.02$ & $0.99\pm0.01$ & $0.62\pm0.08$ & \\
	& MRS & $0.87\pm0.01$ & $0.839\pm0.01$ & $0.753\pm0.02$ & $0.989\pm0.01$ & $0.656\pm0.08$ & \\
	& FW-MRS-RF-Abs & $0.841\pm0.01$ & $0.822\pm0.02$ & $0.75\pm0.02$ & $0.989\pm0.01$ & $0.559\pm0.07$ & \\
	& FW-MRS-SVM-Abs & $0.855\pm0.01$ & $0.827\pm0.01$ & $0.743\pm0.02$ & $0.986\pm0.01$ & $0.536\pm0.09$ & \\
	& FW-MRS-RF-Signed & $0.873\pm0.01$ & $0.84\pm0.01$ & $0.751\pm0.02$ & $0.989\pm0.01$ & $0.701\pm0.07$ & \\
	& FW-MRS-SVM-Signed & $0.868\pm0.01$ & $0.834\pm0.01$ & $0.75\pm0.02$ & $0.987\pm0.01$ & $0.71\pm0.07$ & \\
	& FW-MRS-RF-DA & $0\pm0$ & $0\pm0$ & $0.749\pm0.02$ & $0.987\pm0.01$ & $0.706\pm0.07$ & \\
	& MRS-DA & $0\pm0$ & $0\pm0$ & $0.751\pm0.02$ & $0.988\pm0.01$

In [124]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auprc_values = []
            std_auprc_values = []
            for dataset in datasets:
                try:
                    mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                    mean_auprc_values.append(np.round(mean_auprc, 3))

                    std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                    std_auprc_values.append(np.round(std_auprc, 2))
                except IndexError:
                    mean_auprc_values.append(0)
                    std_auprc_values.append(0)

            print(f"\t& {method} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.827\pm0.02$ & $0.789\pm0.02$ & $0.456\pm0.03$ & $0.994\pm0.0$ & $0.804\pm0.05$ & \\
	& PSA & $0.824\pm0.02$ & $0.781\pm0.02$ & $0.458\pm0.03$ & $0.994\pm0.0$ & $0.794\pm0.06$ & \\
	& KMM & $0.809\pm0.02$ & $0.764\pm0.02$ & $0.442\pm0.04$ & $0.995\pm0.0$ & $0.782\pm0.05$ & \\
	& MRS & $0.827\pm0.02$ & $0.79\pm0.02$ & $0.457\pm0.03$ & $0.995\pm0.0$ & $0.796\pm0.05$ & \\
	& FW-MRS-RF-Abs & $0.788\pm0.02$ & $0.774\pm0.02$ & $0.454\pm0.04$ & $0.995\pm0.0$ & $0.742\pm0.05$ & \\
	& FW-MRS-SVM-Abs & $0.81\pm0.02$ & $0.779\pm0.02$ & $0.436\pm0.05$ & $0.993\pm0.0$ & $0.73\pm0.06$ & \\
	& FW-MRS-RF-Signed & $0.831\pm0.01$ & $0.79\pm0.02$ & $0.457\pm0.03$ & $0.994\pm0.0$ & $0.815\pm0.05$ & \\
	& FW-MRS-SVM-Signed & $0.825\pm0.02$ & $0.785\pm0.02$ & $0.453\pm0.03$ & $0.994\pm0.0$ & $0.813\pm0.05$ & \\
	& FW-MRS-RF-DA & $0\pm0$ & $0\pm0$ & $0.452\pm0.04$ & $0.994\pm0.0$ & $0.816\pm0.05$ & \\
	& MRS-DA & $0\pm0$ & $0\pm0$ & $0.455\pm0.04$ & $0.994\pm0.0$ & $0.

In [125]:
result_df["Rank AUROC"] = result_df.groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS-RF-Abs,6.800000,6.800000
FW-MRS-RF-DA,6.000000,6.000000
FW-MRS-RF-Signed,2.600000,2.000000
FW-MRS-SVM-Abs,8.800000,8.600000
FW-MRS-SVM-DA,10.000000,9.000000
FW-MRS-SVM-Signed,4.800000,5.200000
KMM,6.400000,6.800000
MRS,2.800000,3.000000
MRS-DA,5.666667,5.333333
